# ch03_reservoir_fluids_pvt — 3.10 NeqSim implementation

From chapter `ch03_reservoir_fluids_pvt`, section: **3.10 NeqSim implementation**.

This companion notebook reproduces or expands the textbook code block. Run all cells from top to bottom; cells that are intentionally illustrative are marked in the code comments.



In [1]:
from neqsim import jneqsim
import numpy as np
import matplotlib.pyplot as plt

# 1. Build a rich gas-condensate fluid (typical NCS Sleipner-like)
T_res, P_res = 363.15, 380.0   # 90 °C, 380 bar
fluid = jneqsim.thermo.system.SystemSrkEos(T_res, P_res)
for name, frac in [
    ("nitrogen", 0.005), ("CO2", 0.018), ("methane", 0.770),
    ("ethane",  0.073), ("propane", 0.038), ("i-butane", 0.005),
    ("n-butane", 0.012), ("i-pentane", 0.005),
    ("n-pentane", 0.005), ("n-hexane", 0.008),
    ("n-heptane", 0.061),       # represents C7+ as one pseudo
]:
    fluid.addComponent(name, frac)
fluid.setMixingRule("classic")

# 2. Phase envelope
ops = jneqsim.thermodynamicoperations.ThermodynamicOperations(fluid)
ops.calcPTphaseEnvelope(False, 1.0)
env = ops.getOperation()

def finite_branch(temperatures, pressures):
  T = np.asarray(list(temperatures), dtype=float)
  P = np.asarray(list(pressures), dtype=float)
  keep = np.isfinite(T) & np.isfinite(P)
  return T[keep], P[keep]

branches = []
for label, get_T, get_P in [
  ("bubble-labelled", env.getBubblePointTemperatures, env.getBubblePointPressures),
  ("dew-labelled", env.getDewPointTemperatures, env.getDewPointPressures),
]:
  T, P = finite_branch(get_T(), get_P())
  if len(T) > 0:
    branches.append((label, T, P))
if not branches:
  raise RuntimeError("Phase-envelope calculation returned no finite points")

# Classify by max temperature -> which is dew, which is bubble.
dew_label, dew_T, dew_P = max(branches, key=lambda item: item[1].max())
cricondenbar = max(P.max() for _, _, P in branches)

print(f"Cricondentherm (K): {dew_T.max():.1f} ({dew_label})")
print(f"Cricondenbar  (bar): {cricondenbar:.1f}")

# 3. CME — depressurise from 380 to 50 bar at fixed T
# Reset the fluid state after the envelope calculation
fluid.setTemperature(T_res); fluid.setPressure(P_res)
ops.TPflash()
pressures = np.linspace(380, 50, 30)
rho_total, n_phases = [], []
for P in pressures:
    fluid.setPressure(P)
    ops.TPflash(); fluid.initProperties()
    rho_total.append(fluid.getDensity("kg/m3"))
    n_phases.append(fluid.getNumberOfPhases())



Cricondentherm (K): 398.4 (dew-labelled)
Cricondenbar  (bar): 200.3
